# AutoGluon Tabular — Essential Functionality

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Innixma/kdd2026_tutorial_materials/blob/main/notebooks/02_autogluon_essentials.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-181717?logo=github)](https://github.com/Innixma/kdd2026_tutorial_materials)
[![Tutorial Website](https://img.shields.io/badge/Tutorial-Website-0a7aca?logo=googlechrome&logoColor=white)](https://kdd26-automl-hands-on.github.io/)

**Taming Structured Data Foundation Models with AutoML — KDD 2026 hands-on tutorial**

*Adapted from the official [AutoGluon Tabular Essentials tutorial](https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html), on the dataset from notebook 01.*

Notebook 01 compared individual models by hand. This notebook shows the AutoML way: how
AutoGluon's `TabularPredictor` produces a highly accurate model in 3 lines of code — and how
its `extreme` preset puts the tabular foundation models you just met to work automatically.

We keep working on *polish_companies_bankruptcy*: predict whether a Polish company goes
bankrupt, from 64 financial-ratio features. Same official benchmark split as notebook 01, so
every score here is directly comparable to the staircase we built there.

> **Runtime**: the default fit takes ~1 minute on CPU; the `extreme` fit at the end wants a
> GPU (any Colab GPU runtime works) and takes ~10 minutes with the time limit set below.

## TabularPredictor

To start, import AutoGluon's `TabularPredictor` and `TabularDataset` classes:

In [1]:
# Installs everything the notebook needs, including the tabular foundation models used
# by the `extreme` preset (fast via uv; a no-op where already present).
import sys
!command -v uv >/dev/null || pip install -q uv
!uv pip install -q --python {sys.executable} "autogluon.tabular[tabarena]" openml

from autogluon.tabular import TabularDataset, TabularPredictor

### Loading the data

`TabularDataset` is a convenience wrapper around a [pandas DataFrame](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.html)
and the same methods can be applied to both. We fetch the dataset from OpenML and split it
with the benchmark task's own first train/test split — the same rows as notebook 01.

In [2]:
import openml

task = openml.tasks.get_task(363694)  # polish_companies_bankruptcy
X, y = task.get_X_and_y(dataset_format="dataframe")
train_idx, test_idx = task.get_train_test_split_indices(repeat=0, fold=0)

full_data = X.copy()
full_data[y.name] = y
train_data = TabularDataset(full_data.iloc[train_idx].reset_index(drop=True))
test_data = TabularDataset(full_data.iloc[test_idx].reset_index(drop=True))
print(f"train: {train_data.shape}, test: {test_data.shape}")
train_data.head()

train: (3940, 65), test: (1970, 65)


,net_profit_to_total_assets,total_liabilities_to_total_assets,working_capital_to_total_assets,current_assets_to_short_term_liabilities,liquidity_days_ratio,retained_earnings_to_total_assets,ebit_to_total_assets,book_value_equity_to_total_liabilities,sales_to_total_assets,equity_to_total_assets,...,gross_margin,adjusted_liquidity_ratio,total_costs_to_total_sales,long_term_liabilities_to_equity,inventory_turnover_ratio,receivables_turnover_ratio,short_term_liabilities_days_ratio,sales_to_short_term_liabilities,sales_to_fixed_assets,company_bankrupt
0,0.087072,0.41804,0.046747,1.1118,-7.4579,0.0000,0.10750,1.3921,3.4468,0.58196,...,0.017899,0.14962,0.96930,0.00000,30.7730,10.8200,44.268,8.2453,6.4401,No
1,0.179500,0.11163,0.496930,9.8052,183.8800,0.0000,0.22159,7.9578,0.6300,0.88837,...,0.304560,0.20206,0.66835,0.00000,2.1958,2.9368,32.697,11.1630,1.4106,No
2,0.122530,0.43385,0.219230,1.5942,25.4650,0.2746,0.12391,1.2714,1.0810,0.55160,...,0.074912,0.22213,0.92509,0.11762,16.0230,7.2644,76.443,4.7748,4.2781,No
3,0.158820,0.12695,0.585750,5.6139,42.7190,0.4095,0.19634,6.8723,1.1498,0.87246,...,0.130270,0.18204,0.86973,0.00000,4.0006,9.1458,27.969,13.0500,5.7667,No
4,0.133520,0.38931,0.558480,2.6438,78.7650,0.0000,0.13352,1.5686,1.6678,0.61069,...,0.697750,0.21864,0.90531,0.00000,7.8548,5.2151,74.355,4.9089,16.3870,No


Each row corresponds to one company; the columns are financial ratios from its annual report
(profitability, liquidity, leverage, ...). We predict whether the company goes bankrupt
within the forecasting horizon, indicated by the `company_bankrupt` column. Note the class
imbalance — bankruptcies are rare, which will matter when we choose an evaluation metric.

In [3]:
label = "company_bankrupt"
print(f"Unique classes: {list(train_data[label].unique())}")
print(f"Positive rate: {(train_data[label] == 'Yes').mean():.3f}")

Unique classes: ['No', 'Yes']
Positive rate: 0.070


AutoGluon works with raw data, meaning you don't need to perform any data preprocessing
before fitting AutoGluon. We actively recommend that you avoid performing operations such as
missing value imputation or one-hot-encoding, as AutoGluon has dedicated logic to handle
these situations automatically.

### Pick the evaluation metric first

One decision is worth making *before* fitting: what metric your application actually cares
about. With ~7% positives, accuracy is nearly useless here — a model that answers "No
bankruptcy" for every company is already ~93% accurate. For imbalanced screening problems
the ranking quality matters, so we use `roc_auc`. Passing it to `TabularPredictor` makes
AutoGluon optimize everything — validation, model selection, ensembling — for *that* metric
instead of the default (accuracy for binary classification).

### Training

Now we initialize and fit AutoGluon's TabularPredictor in one line of code:

In [4]:
predictor = TabularPredictor(label=label, eval_metric="roc_auc").fit(train_data)

No path specified. Models will be saved in: "AutogluonModels/ag-20260809_035535"


Verbosity: 2 (Standard Logging)


=================== System Info ===================
AutoGluon Version:  1.6.1.dev0
Python Version:     3.11.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #22~24.04.1-Ubuntu SMP Sat Nov 22 06:23:18 UTC 2025
CPU Count:          192
Pytorch Version:    2.13.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 94.97/94.97 GB
Total GPU Memory:   Free: 94.97 GB, Allocated: 0.00 GB, Total: 94.97 GB
GPU Count:          1
Memory Avail:       1385.34 GB / 1417.32 GB (97.7%)
Disk Space Avail:   1507.81 GB / 9984.00 GB (15.1%)


No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on datasets <100000 samples by using Tabular Foundation Models (TFMs) meta-learned on https://tabarena.ai: Nori, TabICLv2, and TabDPT-Turbo. Every model is free for commercial use. Requires `pip install autogluon.tabular[tabarena]`.
	presets='noncommercial': New in v1.6: 'extreme' plus TabPFN-3, a frontier tabular foundation model created by Prior Labs. Stronger still, but commercial use requires a TabPFN-3 license: https://docs.priorlabs.ai/models#tabpfn-model-license
	presets='best'     : Use this if you do not have a GPU. Maximize accuracy. Use in competi

Using hyperparameters preset: hyperparameters='default'


...

	0.9574	 = Validation score   (roc_auc)


	0.02s	 = Training   runtime


	0.0s	 = Validation runtime


AutoGluon training complete, total runtime = 73.42s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 57257.0 rows/s (500 batch size)


TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/home/nick_priorlabs_ai/workspace_tabpfn_plus/code/kdd2026_tutorial_materials/notebooks/AutogluonModels/ag-20260809_035535")


That's it! We now have a TabularPredictor that is able to make predictions on new data.

### Prediction

We can now use our trained models to make predictions on the held-out test companies:

In [5]:
y_pred = predictor.predict(test_data)
y_pred.head()  # Predictions

0    No
1    No
2    No
3    No
4    No
Name: company_bankrupt, dtype: object

In [6]:
y_pred_proba = predictor.predict_proba(test_data)
y_pred_proba.head()  # Prediction probabilities

,No,Yes
0,0.991585,0.008415
1,0.990420,0.009580
2,0.998977,0.001023
3,0.975970,0.024030
4,0.966486,0.033514


### Evaluation

Next, we can evaluate the predictor on the (labeled) test data:

In [7]:
predictor.evaluate(test_data)

{'roc_auc': np.float64(0.9686878568221181),
 'accuracy': 0.9715736040609138,
 'balanced_accuracy': np.float64(0.8111368593238822),
 'mcc': np.float64(0.7555277141833062),
 'f1': 0.7522123893805309,
 'precision': 0.9444444444444444,
 'recall': 0.625}

`evaluate` leads with the metric the predictor optimizes (`roc_auc`), alongside auxiliary
metrics. Note how high `accuracy` looks despite the mediocre `recall` — the majority-class
trap from the metric discussion above; had we optimized accuracy, the leaderboard below
would rank models by a number that barely reflects screening quality.

We can also evaluate each model individually:

In [8]:
predictor.leaderboard(test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.968688,0.957358,roc_auc,0.018671,0.008733,10.654138,0.002440,0.000243,0.015931,2,True,12
1,CatBoost,0.966587,0.956498,roc_auc,0.005330,0.005524,2.012499,0.005330,0.005524,2.012499,1,True,5
2,XGBoost,0.961996,0.944578,roc_auc,0.010900,0.002965,8.625708,0.010900,0.002965,8.625708,1,True,9
3,LightGBMXT,0.961503,0.935300,roc_auc,0.011349,0.001016,6.381948,0.011349,0.001016,6.381948,1,True,1
4,LightGBM,0.953681,0.930753,roc_auc,0.011978,0.010927,13.477103,0.011978,0.010927,13.477103,1,True,2
5,RandomForestEntr,0.937580,0.896866,roc_auc,0.069075,0.064811,0.356067,0.069075,0.064811,0.356067,1,True,4
6,RandomForestGini,0.934575,0.889708,roc_auc,0.064111,0.064185,0.424072,0.064111,0.064185,0.424072,1,True,3
7,LightGBMLarge,0.922509,0.901935,roc_auc,0.002340,0.000634,8.147668,0.002340,0.000634,8.147668,1,True,11
8,NeuralNetTorch,0.882838,0.894992,roc_auc,0.042981,0.016121,21.694855,0.042981,0.016121,21.694855,1,True,10
9,ExtraTreesGini,0.872480,0.876098,roc_auc,0.069920,0.117632,0.409922,0.069920,0.117632,0.409922,1,True,6


### Loading a trained predictor

The predictor is saved to disk automatically; you can load it in a new session (or on a new
machine) by pointing `TabularPredictor.load()` at its path:

In [9]:
predictor.path  # The path on disk where the predictor is saved

'/home/nick_priorlabs_ai/workspace_tabpfn_plus/code/kdd2026_tutorial_materials/notebooks/AutogluonModels/ag-20260809_035535'

In [10]:
# predictor = TabularPredictor.load(predictor.path)

## Description of fit()

Since there are only two possible values of the `company_bankrupt` variable, this was a
binary classification problem. AutoGluon infers that automatically (along with the type of
each feature, missing-data handling, and rescaling); had we not passed `eval_metric`, it
would also have defaulted the metric to accuracy — which is exactly what we did not want
here.

We did not specify separate validation data, so AutoGluon chose a train/validation split
automatically. Rather than a single model, AutoGluon trains multiple models and ensembles
them together to obtain superior predictive performance — no hyperparameters for you to
specify.

We can view what properties AutoGluon automatically inferred about our prediction task:

In [11]:
print("AutoGluon infers problem type is: ", predictor.problem_type)
print("AutoGluon identified the following types of features:")
print(predictor.feature_metadata)

AutoGluon infers problem type is:  binary
AutoGluon identified the following types of features:
('float', []) : 64 | ['net_profit_to_total_assets', 'total_liabilities_to_total_assets', 'working_capital_to_total_assets', 'current_assets_to_short_term_liabilities', 'liquidity_days_ratio', ...]


To better understand our trained predictor, we can estimate the overall importance of each
feature via permutation importance — how much the score would drop if the feature's values
were shuffled:

In [12]:
predictor.feature_importance(test_data).head(10)

Computing feature importance via permutation shuffling for 64 features using 1970 rows with 5 shuffle sets...


	5.94s	= Expected runtime (1.19s per shuffle set)


	1.66s	= Actual runtime (Completed 5 of 5 shuffle sets)


,importance,stddev,p_value,n,p99_high,p99_low
operating_profit_to_financial_expenses,0.103852,0.011386,0.000017,5,0.127297,0.080407
sales_growth_ratio,0.037389,0.004986,0.000037,5,0.047656,0.027123
operating_expenses_to_total_liabilities,0.018568,0.004643,0.000433,5,0.028128,0.009007
current_assets_minus_inventory_to_short_term_liabilities,0.009929,0.001087,0.000017,5,0.012167,0.007692
sales_profit_to_sales,0.004716,0.000867,0.000131,5,0.006501,0.002931
gross_margin,0.003908,0.000736,0.000144,5,0.005423,0.002394
three_year_gross_profit_to_total_assets,0.003317,0.000966,0.000772,5,0.005305,0.001329
sales_profit_to_total_assets,0.002785,0.000934,0.001313,5,0.004707,0.000863
receivables_plus_inventory_turnover_days,0.002403,0.001301,0.007239,5,0.005081,-0.000275
total_costs_to_total_sales,0.001529,0.001339,0.031534,5,0.004286,-0.001228


Negative `importance` values mean the model may improve if re-fit without that feature.

When we call `predict()`, AutoGluon automatically predicts with the model that displayed the
best performance on validation data (i.e. the weighted ensemble):

In [13]:
predictor.model_best

'WeightedEnsemble_L2'

We can instead specify which model to use for predictions like this:

```python
predictor.predict(test_data, model="LightGBM")
```

You can get the list of trained models via `.leaderboard()` or `.model_names()`:

In [14]:
predictor.model_names()

['LightGBMXT',
 'LightGBM',
 'RandomForestGini',
 'RandomForestEntr',
 'CatBoost',
 'ExtraTreesGini',
 'ExtraTreesEntr',
 'NeuralNetFastAI',
 'XGBoost',
 'NeuralNetTorch',
 'LightGBMLarge',
 'WeightedEnsemble_L2']

## Presets

The scores above used AutoGluon's default preset (`medium`) and default metric. For serious
usage, pick a preset deliberately:

| Preset  | Model Quality                                        | Use Cases | Fit Time (Ideal) | Inference Time (vs medium) | Disk Usage |
|:--------|:-----------------------------------------------------|:----------|:-----------------|:---------------------------|:-----------|
| extreme | **Far better** than best on datasets <100000 samples | (New in v1.6) The absolute cutting edge. Incorporates recent tabular foundation models Nori, TabICLv2, and TabDPT-Turbo. Every model is free for commercial use. Requires a GPU for best results. | 1x | 8x | 2x |
| noncommercial | **Far better** than best on datasets <100000 samples | (New in v1.6) `extreme` plus TabPFN-3, a frontier tabular foundation model created by Prior Labs. Commercial use of TabPFN-3 requires a license from Prior Labs ([license FAQ](https://docs.priorlabs.ai/models#tabpfn-model-license)). Requires a GPU for best results. | 1x | 8x | 2x |
| best    | State-of-the-art (SOTA), much better than high       | When accuracy is what matters and no GPU is available. Has been used to win numerous Kaggle competitions. | 16x+ | 32x+ | 16x+ |
| high    | Better than good                                     | A very powerful, portable solution with fast inference. | 16x+ | 4x | 2x |
| good    | Stronger than any other AutoML framework             | Highly portable, very fast inference. | 16x | 2x | 0.1x |
| medium  | Competitive with other top AutoML frameworks         | Initial prototyping, establishing a performance baseline. | 1x | 1x | 1x |

**If you have a GPU, start with `extreme`.** It is meta-learned from
[TabArena](https://tabarena.ai) and is far better than `best` on datasets below 100,000
samples, while training faster and producing a smaller predictor. Install its dependencies
with `pip install autogluon[tabarena]`. Without a GPU, start with `best`.

## Maximizing predictive performance

**Note:** You should not call `fit()` with entirely default arguments if you are
benchmarking AutoGluon-Tabular or hoping to maximize its accuracy! To get the best
predictive accuracy with AutoGluon, you should generally use it like this:

In [15]:
time_limit = 600  # for quick demonstration only; set this to the longest time you are willing to wait (in seconds)
predictor = TabularPredictor(label, eval_metric="roc_auc").fit(
    train_data, time_limit=time_limit, presets="extreme"
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260809_035654"


Preset alias specified: 'extreme' maps to 'extreme_quality'.


Verbosity: 2 (Standard Logging)


=================== System Info ===================
AutoGluon Version:  1.6.1.dev0
Python Version:     3.11.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #22~24.04.1-Ubuntu SMP Sat Nov 22 06:23:18 UTC 2025
CPU Count:          192
Pytorch Version:    2.13.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 94.97/94.97 GB
Total GPU Memory:   Free: 94.97 GB, Allocated: 0.00 GB, Total: 94.97 GB
GPU Count:          1
Memory Avail:       1379.01 GB / 1417.32 GB (97.3%)
Disk Space Avail:   1507.75 GB / 9984.00 GB (15.1%)


Presets specified: ['extreme']


...

	0.9951	 = Validation score   (roc_auc)


	0.04s	 = Training   runtime


	0.0s	 = Validation runtime


AutoGluon training complete, total runtime = 44.95s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 507.8 rows/s (493 batch size)


TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/home/nick_priorlabs_ai/workspace_tabpfn_plus/code/kdd2026_tutorial_materials/notebooks/AutogluonModels/ag-20260809_035654")


In [16]:
predictor.leaderboard(test_data)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.995534,0.995146,roc_auc,1.887962,3.212266,23.053745,0.002284,0.000428,0.037439,2,True,4
1,PrepLightGBM_BAG_L1,0.994427,0.994162,roc_auc,1.256707,0.650216,13.676744,1.256707,0.650216,13.676744,1,True,3
2,TabICLv2_BAG_L1,0.984356,0.980613,roc_auc,0.628971,2.561622,9.339562,0.628971,2.561622,9.339562,1,True,1
3,TabDPT-Turbo_BAG_L1,0.965222,0.957186,roc_auc,1.243964,1.383117,6.180173,1.243964,1.383117,6.180173,1,True,2


This command implements the following strategy to maximize accuracy:

- Specify `presets="extreme"`, which fits a portfolio of tabular foundation models and
  gradient-boosted trees — meta-learned from TabArena — and ensembles them with
  stacking/bagging. The default `presets="medium"` produces less accurate models but
  facilitates faster prototyping.
- Provide `eval_metric` to `TabularPredictor()` if you know what metric will be used to
  evaluate predictions in your application, as we did from the very first fit (other options
  include `f1`, `log_loss`, `mean_absolute_error`, ...).
- Include all your data in `train_data` and do not provide `tuning_data` (AutoGluon will
  split the data more intelligently to fit its needs).
- Do not specify the `hyperparameter_tune_kwargs` argument (counterintuitively,
  hyperparameter tuning is not the best way to spend a limited training budget — model
  ensembling is often superior, and notebook 01's tuning-trajectory figures show why).
- Do not specify the `hyperparameters` argument (allow AutoGluon to adaptively select which
  models/hyperparameters to use).
- Set `time_limit` to the longest amount of time you are willing to wait.

### Where this lands on notebook 01's staircase

On this exact split, notebook 01 measured: naive XGBoost **0.9628** → AutoGluon-bagged
XGBoost **0.9670** → a single TabICLv2 **0.9838**; the TabArena artifacts put a bagged TabFM
at **0.9952**. The `extreme` leaderboard above shows what an automatically composed
portfolio of foundation models and trees achieves with one `fit()` call — check the
`score_test` of the best model against those numbers.

**Next**: [notebook 03](https://colab.research.google.com/github/Innixma/kdd2026_tutorial_materials/blob/main/notebooks/03_tfm_zoo.ipynb) puts the whole TFM zoo behind a single
AutoGluon predictor.